In [4]:
import pandas as pd
import sqlite3

# Your helper function — copy this into every new notebook
def run_sql(query):
    conn = sqlite3.connect(r'C:/Users/Soumili Nag/Downloads/projects/NovaPay-AML-Analysis/data/novapay_aml.db')
    result = pd.read_sql_query(query, conn)
    conn.close()
    return result

print("✅ Connected to NovaPay AML database")

✅ Connected to NovaPay AML database


# ── QUERY 1: Executive Fraud Overview ──────────────────────────────

In [7]:
# What this tells you: The system is missing almost all fraud (false negatives will be huge). 
# This is your core finding — the existing rule-based system is broken.

result_q1 = run_sql("""
    SELECT
        COUNT(*)                                      AS total_transactions,
        SUM(isFraud)                                  AS confirmed_fraud,
        SUM(isFlaggedFraud)                           AS system_flagged,
        ROUND(SUM(isFraud) * 100.0 / COUNT(*), 4)    AS fraud_rate_pct,
        
        -- False Positive = system flagged it BUT it was NOT actually fraud
        SUM(CASE WHEN isFlaggedFraud = 1 AND isFraud = 0 THEN 1 ELSE 0 END) AS false_positives,
        
        -- False Negative = system MISSED it (fraud but not flagged)
        SUM(CASE WHEN isFlaggedFraud = 0 AND isFraud = 1 THEN 1 ELSE 0 END) AS false_negatives,
        
        ROUND(SUM(amount), 2)                         AS total_volume
    FROM transactions
""")

print("📊 NOVAPAY FINANCIAL — AML FRAUD OVERVIEW")
print("=" * 50)
result_q1

📊 NOVAPAY FINANCIAL — AML FRAUD OVERVIEW


,total_transactions,confirmed_fraud,system_flagged,fraud_rate_pct,false_positives,false_negatives,total_volume
0,6362620,8213,16,0.1291,0,8197,1.144393e+12


# ── QUERY 2: Fraud Rate by Transaction Type ─────────────────────────

In [8]:
# KEY INSIGHT: Only CASH_OUT and TRANSFER are fraud vectors

result_q2 = run_sql("""
    SELECT
        type,
        COUNT(*)                                        AS total_transactions,
        SUM(isFraud)                                    AS fraud_count,
        ROUND(SUM(isFraud) * 100.0 / COUNT(*), 3)      AS fraud_rate_pct,
        ROUND(SUM(CASE WHEN isFraud=1 THEN amount END), 2) AS total_fraud_amount,
        ROUND(AVG(CASE WHEN isFraud=1 THEN amount END), 2) AS avg_fraud_amount
    FROM transactions
    GROUP BY type
    ORDER BY fraud_count DESC
""")

print("📊 FRAUD BREAKDOWN BY TRANSACTION TYPE")
print("=" * 50)
result_q2

📊 FRAUD BREAKDOWN BY TRANSACTION TYPE


,type,total_transactions,fraud_count,fraud_rate_pct,total_fraud_amount,avg_fraud_amount
0,CASH_OUT,2237500,4116,0.184,5.989202e+09,1455102.59
1,TRANSFER,532909,4097,0.769,6.067213e+09,1480891.67
2,PAYMENT,2151495,0,0.000,NaN,NaN
3,DEBIT,41432,0,0.000,NaN,NaN
4,CASH_IN,1399284,0,0.000,NaN,NaN


# ── QUERY 3: High-Risk Senders (Top Fraudulent Originators) ─────────

In [9]:
# These are accounts that initiated the most fraud — your SAR candidates

result_q3 = run_sql("""
    SELECT
        nameOrig                                    AS customer_id,
        COUNT(*)                                    AS total_transactions,
        SUM(isFraud)                                AS fraud_transactions,
        ROUND(SUM(amount), 2)                       AS total_amount_sent,
        ROUND(SUM(CASE WHEN isFraud=1 THEN amount ELSE 0 END), 2) AS fraud_amount,
        type                                        AS transaction_type,
        
        -- Risk Tier based on fraud count
        CASE 
            WHEN SUM(isFraud) >= 3 THEN 'CRITICAL'
            WHEN SUM(isFraud) = 2  THEN 'HIGH'
            WHEN SUM(isFraud) = 1  THEN 'MEDIUM'
            ELSE 'LOW'
        END AS risk_tier
        
    FROM transactions
    WHERE isFraud = 1
    GROUP BY nameOrig
    ORDER BY fraud_transactions DESC, fraud_amount DESC
    LIMIT 30
""")

print("🚨 TOP HIGH-RISK CUSTOMERS (SAR CANDIDATES)")
print("=" * 50)
result_q3

🚨 TOP HIGH-RISK CUSTOMERS (SAR CANDIDATES)


,customer_id,total_transactions,fraud_transactions,total_amount_sent,fraud_amount,transaction_type,risk_tier
0,C1004068843,1,1,10000000.0,10000000.0,TRANSFER,MEDIUM
1,C1014298205,1,1,10000000.0,10000000.0,TRANSFER,MEDIUM
2,C1016892734,1,1,10000000.0,10000000.0,CASH_OUT,MEDIUM
3,C1028530067,1,1,10000000.0,10000000.0,TRANSFER,MEDIUM
4,C1036572575,1,1,10000000.0,10000000.0,CASH_OUT,MEDIUM
5,C1041060645,1,1,10000000.0,10000000.0,CASH_OUT,MEDIUM
6,C1045409609,1,1,10000000.0,10000000.0,TRANSFER,MEDIUM
7,C1046281837,1,1,10000000.0,10000000.0,CASH_OUT,MEDIUM
8,C1049094143,1,1,10000000.0,10000000.0,CASH_OUT,MEDIUM
9,C1057439889,1,1,10000000.0,10000000.0,TRANSFER,MEDIUM


# ── QUERY 4: Transaction Velocity Analysis ──────────────────────────

In [11]:
# Fraudsters move money fast — many transactions in short time windows
# 'step' = hour. We look for accounts with high activity in few hours.

result_q4 = run_sql("""
    SELECT
        nameOrig                            AS customer_id,
        COUNT(*)                            AS transaction_count,
        COUNT(DISTINCT step)                AS active_hours,
        ROUND(COUNT(*) * 1.0 / 
              COUNT(DISTINCT step), 2)      AS transactions_per_hour,
        MIN(step)                           AS first_seen_hour,
        MAX(step)                           AS last_seen_hour,
        (MAX(step) - MIN(step))             AS hour_span,
        ROUND(SUM(amount), 2)               AS total_amount,
        SUM(isFraud)                        AS fraud_count,
        
        -- Velocity Risk Flag
        CASE
            WHEN COUNT(*) * 1.0 / COUNT(DISTINCT step) > 3 THEN 'HIGH VELOCITY ⚠️'
            WHEN COUNT(*) * 1.0 / COUNT(DISTINCT step) > 1.5 THEN 'MODERATE'
            ELSE 'NORMAL'
        END AS velocity_risk
        
    FROM transactions
    GROUP BY nameOrig
    HAVING transaction_count > 3
    ORDER BY transactions_per_hour DESC
    LIMIT 25
""")

print("⚡ TRANSACTION VELOCITY ANALYSIS")
print("=" * 50)
result_q4

⚡ TRANSACTION VELOCITY ANALYSIS


,customer_id,transaction_count,active_hours,transactions_per_hour,first_seen_hour,last_seen_hour,hour_span,total_amount,fraud_count,velocity_risk


# ── QUERY 5: Balance Wipe-Out Pattern ───────────────────────────────

In [13]:
# Classic fraud pattern: account sends ALL its money and balance goes to 0
# This is one of the strongest fraud signals in AML

result_q5 = run_sql("""
    SELECT
        nameOrig                            AS customer_id,
        type,
        ROUND(amount, 2)                    AS amount,
        ROUND(oldbalanceOrg, 2)             AS balance_before,
        ROUND(newbalanceOrig, 2)            AS balance_after,
        isFraud,
        
        -- Balance wipe = sent almost all their money
        CASE
            WHEN oldbalanceOrg > 0 AND newbalanceOrig = 0 THEN 'FULL WIPE ⚠️'
            WHEN oldbalanceOrg > 0 AND 
                 (oldbalanceOrg - newbalanceOrig) / oldbalanceOrg > 0.9 
                 THEN 'NEAR WIPE (>90%)'
            ELSE 'NORMAL'
        END AS balance_pattern,
        
        -- Amount matches balance exactly (strong fraud signal)
        CASE
            WHEN ROUND(amount,2) = ROUND(oldbalanceOrg,2) THEN 'YES ⚠️'
            ELSE 'NO'
        END AS amount_equals_balance
        
    FROM transactions
    WHERE type IN ('TRANSFER', 'CASH_OUT')
      AND oldbalanceOrg > 0
      AND newbalanceOrig = 0
    ORDER BY amount DESC
    LIMIT 30
""")

print("💸 BALANCE WIPE-OUT PATTERN DETECTION")
print("=" * 50)
result_q5

💸 BALANCE WIPE-OUT PATTERN DETECTION


,customer_id,type,amount,balance_before,balance_after,isFraud,balance_pattern,amount_equals_balance
0,C208486812,TRANSFER,57787800.93,42228.75,0.0,0,FULL WIPE ⚠️,NO
1,C1483754162,TRANSFER,51141938.17,310058.79,0.0,0,FULL WIPE ⚠️,NO
2,C539714486,TRANSFER,47504216.38,17047.14,0.0,0,FULL WIPE ⚠️,NO
3,C240100497,TRANSFER,45372634.63,5819.77,0.0,0,FULL WIPE ⚠️,NO
4,C1835360608,TRANSFER,42183808.56,266493.90,0.0,0,FULL WIPE ⚠️,NO
5,C2080983471,TRANSFER,38919599.87,40426.97,0.0,0,FULL WIPE ⚠️,NO
6,C1741076564,TRANSFER,38874009.46,435622.96,0.0,0,FULL WIPE ⚠️,NO
7,C2078977034,TRANSFER,38448653.52,172587.81,0.0,0,FULL WIPE ⚠️,NO
8,C2133858182,TRANSFER,37663153.97,1768.91,0.0,0,FULL WIPE ⚠️,NO
9,C102829469,TRANSFER,37387628.10,862621.88,0.0,0,FULL WIPE ⚠️,NO


# ── QUERY 6: Fraud Volume by Hour ───────────────────────────────────

In [14]:
# This becomes your time-series line chart in Tableau

result_q6 = run_sql("""
    SELECT
        step                                        AS hour,
        COUNT(*)                                    AS total_transactions,
        SUM(isFraud)                                AS fraud_count,
        ROUND(SUM(CASE WHEN isFraud=1 THEN amount ELSE 0 END), 2) AS fraud_amount,
        ROUND(SUM(isFraud) * 100.0 / COUNT(*), 4)  AS fraud_rate_pct
    FROM transactions
    GROUP BY step
    ORDER BY step
""")

print("📅 HOURLY FRAUD TREND (30-DAY WINDOW)")
print("=" * 50)
result_q6.head(20)  # show first 20 rows, full data saved below

📅 HOURLY FRAUD TREND (30-DAY WINDOW)


,hour,total_transactions,fraud_count,fraud_amount,fraud_rate_pct
0,1,2708,16,3740247.01,0.5908
1,2,1014,8,4186592.48,0.7890
2,3,552,4,66832.74,0.7246
3,4,565,10,26400274.90,1.7699
4,5,665,6,381841.54,0.9023
5,6,1660,22,974869.68,1.3253
6,7,6837,12,12414694.06,0.1755
7,8,21097,12,1589040.41,0.0569
8,9,37628,19,11476630.22,0.0505
9,10,35991,11,6935977.72,0.0306


# ── QUERY 7: System Alert Quality Analysis ──────────────────────────

In [15]:
# This is your KEY AML finding: the rule-based system is nearly useless
# False Positives waste analyst time. False Negatives = missed fraud = $$$

result_q7 = run_sql("""
    SELECT
        -- Confusion matrix breakdown
        SUM(CASE WHEN isFlaggedFraud=1 AND isFraud=1 THEN 1 ELSE 0 END) AS true_positive,
        SUM(CASE WHEN isFlaggedFraud=1 AND isFraud=0 THEN 1 ELSE 0 END) AS false_positive,
        SUM(CASE WHEN isFlaggedFraud=0 AND isFraud=0 THEN 1 ELSE 0 END) AS true_negative,
        SUM(CASE WHEN isFlaggedFraud=0 AND isFraud=1 THEN 1 ELSE 0 END) AS false_negative,
        
        -- Precision: of all flagged, how many were real fraud?
        ROUND(
            SUM(CASE WHEN isFlaggedFraud=1 AND isFraud=1 THEN 1.0 ELSE 0 END) /
            NULLIF(SUM(isFlaggedFraud), 0) * 100, 2
        ) AS precision_pct,
        
        -- Recall: of all real fraud, how many did we catch?
        ROUND(
            SUM(CASE WHEN isFlaggedFraud=1 AND isFraud=1 THEN 1.0 ELSE 0 END) /
            NULLIF(SUM(isFraud), 0) * 100, 2
        ) AS recall_pct
        
    FROM transactions
""")

print("🎯 SYSTEM ALERT QUALITY — CONFUSION MATRIX")
print("=" * 50)
result_q7

🎯 SYSTEM ALERT QUALITY — CONFUSION MATRIX


,true_positive,false_positive,true_negative,false_negative,precision_pct,recall_pct
0,16,0,6354407,8197,100.0,0.19


In [17]:
import pandas as pd

# Recreate alert quality in the RIGHT format for Tableau
alert_quality_tableau = pd.DataFrame([
    {'Metric': 'True Negative',  'Value': 6354407, 'Category': 'Correct'},
    {'Metric': 'False Negative', 'Value': 8197,    'Category': 'Error'},
    {'Metric': 'True Positive',  'Value': 16,      'Category': 'Correct'},
    {'Metric': 'False Positive', 'Value': 0,       'Category': 'Error'},
    {'Metric': 'Precision %',    'Value': 100.0,   'Category': 'Performance'},
    {'Metric': 'Recall %',       'Value': 0.19,    'Category': 'Performance'},
])

alert_quality_tableau.to_csv('../data/exports/07_alert_quality.csv', index=False)
print("✅ Fixed! Preview:")
print(alert_quality_tableau)

✅ Fixed! Preview:
           Metric       Value     Category
0   True Negative  6354407.00      Correct
1  False Negative     8197.00        Error
2   True Positive       16.00      Correct
3  False Positive        0.00        Error
4     Precision %      100.00  Performance
5        Recall %        0.19  Performance


# ── EXPORT: Save all query results for Tableau ──────────────────────

In [16]:
import os
os.makedirs('../data/exports', exist_ok=True)

result_q1.to_csv('../data/exports/01_fraud_overview.csv', index=False)
result_q2.to_csv('../data/exports/02_fraud_by_type.csv', index=False)
result_q3.to_csv('../data/exports/03_high_risk_customers.csv', index=False)
result_q4.to_csv('../data/exports/04_velocity_analysis.csv', index=False)
result_q5.to_csv('../data/exports/05_balance_wipeout.csv', index=False)
result_q6.to_csv('../data/exports/06_hourly_trend.csv', index=False)
result_q7.to_csv('../data/exports/07_alert_quality.csv', index=False)

print("✅ All 7 query results exported to /data/exports/")
print("   These CSVs will be imported into Tableau for your dashboard.")

✅ All 7 query results exported to /data/exports/
   These CSVs will be imported into Tableau for your dashboard.
